In [0]:
# Parameters
storage_account = "nilaystore"
container = "raw"
file_path = "retail_iot_data.csv"

# Credentials - Replace with actual values (or keep them in a secret scope)
client_id = "<YOUR_CLIENT_ID>"
tenant_id = "YOUR_TENANT_ID"
client_secret = "YOUR_CLIENT_SECRET"

# ---------------------------
# Configurations for ABFS access (no mount needed in UC)
# ---------------------------
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# ---------------------------
# Build ABFS path directly (no /mnt needed)
# ---------------------------
abfs_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{file_path}"

# ---------------------------
# Try reading the file to verify access
# ---------------------------
# Read raw file from ADLS
df = spark.read.format("csv").option("header", "true").load(abfs_path)
display(df)


OrderID,CustomerID,Region,StoreID,Product,Quantity,UnitPrice,SalesAmount,OrderTime,DeviceType,Temperature,DeviceStatus,AnomalyFlag,AnomalyType
db07ec0d-e8e7-408a-846e-c2d330484d4b,CUST-2824,East,Store-009,Mobile,2,322.1,644.2,2023-01-05 23:24:55,Temp_Sensor,23.18,OK,0,null
93d89258-6ef2-4bb8-9a71-cb590cb55f85,CUST-7912,East,Store-001,Laptop,2,503.69,1007.38,2023-01-30 05:24:09,RFID_Scanner,21.93,OK,0,null
f97636e9-3dab-48ad-aad4-0eca034d3111,CUST-9928,South,Store-008,Tablet,5,592.47,2962.35,2023-01-01 07:34:17,RFID_Scanner,22.89,OK,0,null
f20eae76-3e09-4580-8429-af1d6b76209a,CUST-3547,West,Store-011,Laptop,1,790.86,790.86,2023-01-18 10:14:28,POS_Terminal,22.23,OK,0,null
5dbc9dfd-0275-4427-a4b6-089b43465200,CUST-8527,East,Store-013,Laptop,11,621.7,6838.7,2023-01-18 13:20:07,Temp_Sensor,19.35,MISMATCH,1,RFID_Mismatch
c6bd3963-d239-42b7-b414-4e39289bc2da,CUST-5741,East,Store-008,Printer,1,791.25,791.25,2023-01-23 00:15:40,Temp_Sensor,23.84,OK,0,null
02fadf02-d15e-4527-94b1-9cb915a71209,CUST-6820,West,Store-009,Smartwatch,1,1237.81,1237.81,2023-01-09 07:23:27,Temp_Sensor,23.1,OK,0,null
90e7e190-9d09-47a3-a399-98562cf389dc,CUST-7216,North,Store-018,Mobile,3,1693.56,5080.68,2023-01-03 17:10:20,RFID_Scanner,23.75,OK,0,null
50f6c840-47fc-4117-a776-78701a91f199,CUST-7572,North,Store-003,Mobile,5,1758.92,8794.6,2023-01-16 06:37:34,RFID_Scanner,22.59,OK,0,null
77d4cd70-6019-4be0-bce4-51626ae52101,CUST-8517,West,Store-009,Mobile,2,1502.73,3005.46,2023-01-27 03:57:14,POS_Terminal,23.23,OK,0,null


In [0]:
import dlt
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Define schema for the raw CSV data
iot_schema = StructType([
    StructField("OrderID", StringType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("StoreID", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("SalesAmount", DoubleType(), True),
    StructField("OrderTime", StringType(), True),
    StructField("DeviceType", StringType(), True),
    StructField("Temperature", DoubleType(), True),
    StructField("DeviceStatus", StringType(), True),
    StructField("AnomalyFlag", IntegerType(), True),
    StructField("AnomalyType", StringType(), True)
])

# ----------------------------------------------------------------
# Bronze Layer: Ingestion (Raw Data)
# ----------------------------------------------------------------
# This table ingests the raw data stream using Auto Loader.
@dlt.table(
    name="iot_orders_bronze",
    comment="Raw ingested data from ADLS."
)
def iot_orders_bronze():
    # Use spark.readStream with cloudFiles (Auto Loader) for a production-like streaming setup.
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", True)
            .schema(iot_schema)
            .load("abfss://raw@nilaystore.dfs.core.windows.net/incoming")
    )

EVENTHUB_CONNECTION_STRING = "YOUR_EVENT_HUB_CONNECTION_STRING"
EVENT_HUB_NAME = "YOUR_EVENT_HUB_NAME"

@dlt.table(
    name="iot_orders_bronze_eventhub",
    comment="Raw IoT data ingested from Event Hub"
)
def iot_orders_bronze_eventhub():
    # Read stream from Event Hub
    df = (spark.readStream
            .format("eventhubs")
            .option("eventhubs.connectionString", EVENTHUB_CONNECTION_STRING)
            .load()
         )
    
    # Parse Event Hub body (binary) as JSON
    df_parsed = (df.selectExpr("CAST(body AS STRING) as json_str")
                   .select(F.from_json("json_str", iot_schema).alias("data"))
                   .select("data.*")
                )
    
    return df_parsed


The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
DLTImportException                        Traceback (most recent call last)
File <command-7835004079568610>, line 1
----> 1 import dlt
      2 import pyspark.sql.functions as F
      3 from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

File /databricks/python_shell/lib/dbruntime/autoreload/discoverability/hook.py:71, in AutoreloadDiscoverabilityHook._patched_import(self, name, *args, **kwargs)
     65 if not self._should_hint and (
     66     (module := sys.modules.get(absolute_name)) is not None and
     67     (fname := get_allowed_file_name_or_none(module)) is not None and
     68     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     69         absolute_name, float("inf")) and not self._should_hint):
     70     self._should_hint = True
---> 71 module = self._original_builtins_import(name, *args, **kwargs)
     72 if (fname := f

In [0]:
# ----------------------------------------------------------------
# Silver Layer: Transformation (Cleaned Data)
# ----------------------------------------------------------------
# This table cleans the Bronze data and performs type casting and deduplication.
@dlt.table(
    name="nilay_retail_iot_catalog.silver.iot_cleaned_silver",
    comment="Cleaned and validated data with correct types and no duplicates.",
    table_properties={"delta.enableChangeDataFeed": "true"} # Enable Change Data Feed for upserts
)
def iot_cleaned_silver():
    df_bronze = dlt.read("iot_orders_bronze")
    df_bronze_eventhub = dlt.read("iot_orders_bronze_eventhub")

    # Combine both sources using unionByName
    df_combined = df_bronze.unionByName(df_bronze_eventhub, allowMissingColumns=True)
   
    # Renaming columns for clarity and consistency
    df_silver = df_combined.toDF(
        "order_id", "customer_id", "region", "store_id", "product", "quantity",
        "unit_price", "sales_amount", "order_time", "device_type", "temperature",
        "device_status", "anomaly_flag", "anomaly_type"
    )
 
    # Apply data type conversions and transformations
    df_silver = (df_silver
        .withColumn("order_time", F.to_timestamp(F.col("order_time")))
        .withColumn("sales_amount_calculated", F.col("quantity") * F.col("unit_price"))
        .withColumn("is_error", F.when(F.col("device_status") == "ERROR", 1).otherwise(0))
        # Handle nulls in anomaly_type column
        .withColumn("anomaly_type", F.when(F.col("anomaly_type").isNull(), "No Anomaly").otherwise(F.col("anomaly_type")))
    )
 
    # Deduplicate based on OrderID
    # DLT handles deduplication automatically if a primary key is defined in the pipeline settings.
    return df_silver.dropDuplicates(["order_id"])
 

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7835004079568560>, line 5
      1 # ----------------------------------------------------------------
      2 # Silver Layer: Transformation (Cleaned Data)
      3 # ----------------------------------------------------------------
      4 # This table cleans the Bronze data and performs type casting and deduplication.
----> 5 @dlt.table(
      6     name="nilay_retail_iot_catalog.silver.iot_cleaned_silver",
      7     comment="Cleaned and validated data with correct types and no duplicates.",
      8     table_properties={"delta.enableChangeDataFeed": "true"} # Enable Change Data Feed for upserts
      9 )
     10 def iot_cleaned_silver():
     11     # Read from the Bronze table using dlt.read()
     12     df_bronze = dlt.read("iot_orders_bronze")
     14     # Renaming columns for clarity and consistency

NameError: nam

In [0]:
# ----------------------------------------------------------------
# Gold Layer: Aggregation (Final KPIs)
# ----------------------------------------------------------------
# These tables are created from the Silver layer and are ready for Power BI.
 
# KPI 1: Monthly Sales per Region
@dlt.table(
    name="nilay_retail_iot_catalog.gold.monthly_sales_kpis",
    comment="Monthly sales per region for executive reporting."
)
def gold_monthly_sales():
    df_silver = dlt.read("nilay_retail_iot_catalog.silver.iot_cleaned_silver") # Corrected
    return (df_silver
        .withColumn("order_month", F.trunc(F.col("order_time"), "month"))
        .groupBy("order_month", "region", "store_id")
        .agg(
            F.round(F.sum("sales_amount_calculated"), 2).alias("monthly_sales"),
            F.sum("anomaly_flag").alias("monthly_anomalies")
        )
    )
 
# KPI 2: Store Tiers & Anomaly Impact
@dlt.table(
    name="nilay_retail_iot_catalog.gold.store_performance_kpis",
    comment="Store tiers and anomaly impact on sales."
)
def gold_store_performance():
    df_silver = dlt.read("nilay_retail_iot_catalog.silver.iot_cleaned_silver") # Corrected
    store_kpis = (df_silver
        .groupBy("store_id")
        .agg(
            F.sum("sales_amount_calculated").alias("total_sales"),
            F.sum("anomaly_flag").alias("total_anomalies"),
            F.count("order_id").alias("total_orders")
        )
    )
    return (store_kpis
        .withColumn("store_tier",
            F.when(F.col("total_sales") > 10000, "High Performer")
            .when(F.col("total_sales") > 5000, "Medium Performer")
            .otherwise("Low Performer")
        )
        .withColumn("conversion_rate",
            F.when(F.col("total_orders") > 0, F.col("total_anomalies") / F.col("total_orders"))
            .otherwise(F.lit(None))
        )
    )
 
# KPI 3: Weighted Average of Sales vs Anomalies
@dlt.table(
    name="nilay_retail_iot_catalog.gold.weighted_metrics_kpis",
    comment="Aggregated sales and anomaly counts by device type."
)
def gold_weighted_metrics():
    df_silver = dlt.read("nilay_retail_iot_catalog.silver.iot_cleaned_silver") # Corrected
    return (df_silver
        .groupBy("device_type")
        .agg(
            F.round(F.sum("sales_amount_calculated"), 2).alias("total_sales"),
            F.sum("anomaly_flag").alias("total_anomalies")
        )
        .withColumn("sales_per_anomaly",
            F.when(F.col("total_anomalies") > 0, F.col("total_sales") / F.col("total_anomalies"))
            .otherwise(F.lit(None))
        )
    )

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7835004079568577>, line 7
      1 # ----------------------------------------------------------------
      2 # Gold Layer: Aggregation (Final KPIs)
      3 # ----------------------------------------------------------------
      4 # These tables are created from the Silver layer and are ready for Power BI.
      5 
      6 # KPI 1: Monthly Sales per Region
----> 7 @dlt.table(
      8     name="nilay_retail_iot_catalog.gold.monthly_sales_kpis",
      9     comment="Monthly sales per region for executive reporting."
     10 )
     11 def gold_monthly_sales():
     12     df_silver = dlt.read("iot_cleaned_transactions")
     13     return (df_silver
     14         .withColumn("order_month", F.trunc(F.col("order_time"), "month"))
     15         .groupBy("order_month", "region", "store_id")
   (...)
     19         )
     20 